In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model


load_dotenv(override = True)
apikey = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL")
g_apikey = os.getenv("GEMINI_API_KEY")
g_default_model = os.getenv("GEMINI_DEFAULT_MODEL")

if not apikey:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to the .env file.")
if not default_model:
    raise RuntimeError("OPENAI_DEFAULT_MODEL is not set. Add it to the .env file.")

default_model = default_model.strip().lower()
print(f"Using model: {default_model}")
print(f"Gemini API key configured: {bool(g_apikey)}")
print(f"Gemini model: {g_default_model}")

llm = init_chat_model('openai:gpt-6-luna')

Using model: gpt-6-luna
Gemini API key configured: True
Gemini model: gemini-3.5-flash-lite


In [9]:
# 환경 변수 가져오기

from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
os.getenv("TAVILY_API_KEY")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

사용 모델: GPT-6-LUNA


In [10]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""  # Docstring
    return a + b
    
@tool
def get_word_length(word: str) -> int:
    """단어나 문장의 정확한 글자 수(길이)를 계산할 때 사용합니다."""
    return len(word)    
           
print(add.name, "-", add.description)
print(add.args)
print(add.invoke({"a": 5, "b": 7}))   

add - 두 숫자를 더합니다.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
12


In [14]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# 셀을 실행할 때마다 한국 시간 기준 날짜를 계산합니다.
today = datetime.now(ZoneInfo("Asia/Seoul")).date()
today_text = today.strftime("%Y년 %m월 %d일")
today_iso = today.isoformat()

web_search = TavilySearch(max_results=5)

agent = create_agent(
    model=llm,
    tools=[web_search, get_word_length, add],
    system_prompt=(
        f"오늘은 {today_text} (Asia/Seoul 기준)입니다. "
        "날씨 질문에는 검색 도구를 사용하고, 검색 질의에 오늘 날짜를 포함하세요. "
        "검색 결과의 게시일과 예보 대상 날짜를 확인하고, 오늘 날짜와 일치하는 정보만 오늘의 정보로 안내하세요. "
        "날짜가 불분명하거나 오늘 정보가 없으면 오래된 결과를 오늘 정보처럼 말하지 말고, 최신 정보를 확인할 수 없다고 명확히 알리세요. "
        "비교 결과에는 근거가 된 출처 링크를 포함하세요."
    ),
)

result = agent.invoke({
    "messages": [
        (
            "human",
            f"오늘({today_iso}, 한국 시간) 서울과 도쿄의 현재 날씨와 오늘 남은 시간 예보를 검색해 비교해 줘. "
            "각 정보가 오늘 날짜를 대상으로 하는지 확인하고 출처 링크도 알려 줘.",
        )
    ]
})
print(result["messages"][-1].text)


## 서울·도쿄 날씨 비교 — 2026년 10월 15일(목), 현지 시각

검색 결과 중 **예보 대상일이 10월 15일로 명시된 자료**를 기준으로 비교했습니다. 서울·도쿄는 모두 UTC+9라 시각대는 같습니다.

| 도시 | 현재 날씨 | 오늘 남은 시간 예보 |
|---|---|---|
| **서울** | 서울 김포공항 관측소 기준, **오전 9시 19분 현재 약 59°F(15°C)**로 표시됩니다. 다만 페이지에 실린 최신 관측 보고는 **오전 8시**입니다. | 오늘 예보는 대체로 **오후 약 21°C**, 저녁에는 **약 18°C 안팎**입니다. 낮에는 흐려지고, 저녁에는 **약한 빗방울 가능성**이 있다는 예보가 있습니다. |
| **도쿄** | 오늘 날짜의 페이지에 현재 기온 **19°C**가 표시되지만, 그 수치의 **관측 시각은 명시되지 않아** 서울의 시각이 찍힌 관측값과 직접 비교하기는 어렵습니다. | 오늘은 **구름 사이로 해가 나는 날씨**, 최고 **23°C·최저 19°C** 예보입니다. 오후 기온은 약 **21–22°C**, 저녁은 **약 18–19°C** 수준이며, Yahoo! 날씨는 **18–24시 강수확률 0%**로 안내합니다. |

### 확인한 출처와 날짜

- [서울 현재 날씨·시간별/48시간 예보 — timeanddate](https://www.timeanddate.com/weather/south-korea/seoul): 페이지에 **현재 시각 10월 15일 09:19**, 최신 관측 **08:00**, 자료 업데이트 **10월 15일 04:42(서울 시각)**가 표시됩니다. 남은 시간 예보의 흐림·저녁 약한 비 정보도 이 페이지에서 확인했습니다.
- [도쿄 오늘·내일 예보 — Yahoo! Japan Weather](https://weather.yahoo.co.jp/weather/jp/13/4410.html): 예보 발표 시각은 **10월 15일 05:00**, 대상 날짜는 **10월 15일**입니다. 최고·최저기온과 

In [ ]:
import gradio as gr

# 인터페이스에서 이벤트가 발생되면 실행되는 함수
def greet(name):
    return f"Hello {name}!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")

demo.launch(share=True) #server_name="0.0.0.0"

In [76]:
demo.close()      # 이 데모만 종료
gr.close_all()    # 실행 중인 모든 Gradio 데모 종료

Closing server running on port: 7861
Closing server running on port: 7860
Closing server running on port: 7862


In [ ]:
import gradio as gr

# 여기서의 input이 인터페이스의 inputs (순서대로)
def order(name, is_iced, shots):
    drink = "아이스 아메리카노" if is_iced else "따뜻한 아메리카노"
    message = f"{name}님, {drink} (샷 {shots}개) 주문이 접수되었습니다."
    price = 4000 + (shots - 1) * 500   # 기본 4,000원, 샷 추가 시 500원씩
    return message, price              # 리턴은 인터페이스의 outputs (순서대로)

demo = gr.Interface(
    fn=order,
    inputs=[
        gr.Textbox(label="이름"),
        gr.Checkbox(label="아이스로 주문"),
        gr.Slider(1, 5, step=1, value=1, label="샷 개수"),
    ],
    outputs=[
        gr.Textbox(label="주문 확인"),
        gr.Number(label="결제 금액(원)"),
    ]
)

demo.launch(share=True)

In [28]:
gr.close_all()

Closing server running on port: 7860
Closing server running on port: 7863
Closing server running on port: 7864
Closing server running on port: 7861
Closing server running on port: 7862


In [ ]:
import gradio as gr

def respond(message, history):
    return "안녕하세요!"

gr.ChatInterface(fn=respond).launch(share=True)

In [23]:
import gradio as gr

tab1 = gr.Interface(lambda x: x[::-1], inputs="text", outputs="text", title="Reverse")
tab2 = gr.Interface(lambda x: x.upper(), inputs="text", outputs="text", title="Uppercase")

gr.TabbedInterface([tab1, tab2], ["뒤집기", "대문자"]).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import gradio as gr
from time import sleep

def respond(message, history):
    respond = '안녕하세요'

    stream =""
    for char in respond:
        stream += char
        sleep(0.30)
        yield stream

# gr.ChatInterface()
demo = gr.ChatInterface(
    fn=respond, 
    textbox = gr.Textbox(label='메세지 입력', placeholder = '메세지를 입력하세요.')
)

demo.launch(share = True)

In [ ]:
import gradio as gr
from openai import OpenAI

client = OpenAI()
last_id = {}   # 세션별 직전 응답 ID

def respond(message, history, request: gr.Request):
    session_id = request.session_hash

    if not history:                              # 첫 대화/초기화면 이전 대화 연결을 끊음
        last_id.pop(session_id, None)

    stream = client.responses.create(
        model=default_model,
        instructions="당신은 친절한 AI 챗봇입니다.",   # 매 호출마다 다시 넘겨야 함
        input=message,                                 # 이번 메시지만 전달
        previous_response_id=last_id.get(session_id),  # 첫 호출은 None
        reasoning={"effort": "none"},
        stream=True,
    )

    full_response = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            full_response += event.delta
            yield full_response
        elif event.type == "response.completed":
            last_id[session_id] = event.response.id    # 다음 턴에 쓸 ID 저장

gr.ChatInterface(
    fn=respond,
    textbox=gr.Textbox(placeholder="메세지를 입력하세요...", label="입력"),
).launch(share=True)

In [ ]:
import gradio as gr
from openai import OpenAI

client = OpenAI()
last_id = {}   # 세션별 직전 응답 ID

def respond(message, history, request: gr.Request):
    session_id = request.session_hash

    if not history:                              # 첫 대화/초기화면 이전 대화 연결을 끊음
        last_id.pop(session_id, None)

    stream = client.responses.create(
        model=default_model,
        instructions="당신은 친절한 AI 챗봇입니다.",   # 매 호출마다 다시 넘겨야 함
        input=message,                                 # 이번 메시지만 전달
        previous_response_id=last_id.get(session_id),  # 첫 호출은 None
        reasoning={"effort": "none"},
        stream=True,
    )

    full_response = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            full_response += event.delta
            yield full_response
        elif event.type == "response.completed":
            last_id[session_id] = event.response.id    # 다음 턴에 쓸 ID 저장

gr.ChatInterface(
    fn=respond,
    textbox=gr.Textbox(placeholder="메세지를 입력하세요...", label="입력"),
).launch(share=True)

In [ ]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
camping_model = os.getenv("OPENAI_DEFAULT_MODEL")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")
if not camping_model:
    raise RuntimeError(".env 파일에 OPENAI_DEFAULT_MODEL을 설정한 뒤 다시 실행하세요.")

camping_client = OpenAI()
camping_last_response_ids = {}  # Gradio 세션별 이전 응답 ID

def recommend_camping(message, history, request: gr.Request):
    session_id = request.session_hash
    if not history:
        camping_last_response_ids.pop(session_id, None)

    today = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y년 %m월 %d일")
    stream = camping_client.responses.create(
        model=camping_model,
        instructions=(
            f"오늘은 {today}입니다. 한국의 캠핑장 추천 도우미로서 웹 검색을 이용해 답하세요. "
            "요청 조건에 맞는 캠핑장을 추천하고, 위치와 조건에 맞는 이유를 설명하세요. "
            "운영 여부나 시설 정보는 검색 결과로 확인하고, 확인되지 않은 내용은 단정하지 마세요. "
            "가능하면 각 추천의 출처 링크를 함께 제시하세요. 답변은 한국어로 작성하세요."
        ),
        input=message,
        previous_response_id=camping_last_response_ids.get(session_id),
        reasoning={"effort": "none"},
        tools=[{"type": "web_search"}],
        stream=True,
    )

    full_response = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            full_response += event.delta
            yield full_response
        elif event.type == "response.completed":
            camping_last_response_ids[session_id] = event.response.id
        elif event.type == "response.failed":
            error = event.response.error
            raise RuntimeError(error.message if error else "OpenAI 응답 생성에 실패했습니다.")

camping_app = gr.ChatInterface(
    fn=recommend_camping,
    title="캠핑장 추천 챗봇",
    description="예시를 누르거나 원하는 조건을 입력해 캠핑장을 추천받아 보세요.",
    examples=[
        "계곡이 있는 캠핑장을 추천해줘.",
        "가족과 함께 주말을 보낼 수 있는 캠핑장 추천해줘.",
    ],
    examples_label="입력 프롬프트 예시",
    textbox=gr.Textbox(label="캠핑장 조건", placeholder="예: 계곡이 있고 아이와 가기 좋은 곳"),
)
camping_app.launch()  # 기본값은 로컬 실행이며, 외부 공유가 필요하면 share=True로 변경하세요.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

# 고캠핑 API 주소
API_URL = "https://apis.data.go.kr/B551011/GoCamping/searchList"

SERVICE_KEY = os.getenv("GOCAMPING_SERVICE_KEY")


def search_camping(keyword):
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 10,
        "pageNo": 1,
        "MobileOS": "ETC",
        "MobileApp": "AppTest",
        "keyword": keyword,
        "_type": "json"
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=15
    )
    response.raise_for_status()

    data = response.json()

    items = (
        data.get("response", {})
            .get("body", {})
            .get("items", {})
            .get("item", [])
    )

    
    if isinstance(items, dict):
        items = [items]

    return items


def recommend_camping(prompt):
    keyword_response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        캠핑장 검색에 사용할 핵심 키워드 하나만 출력해.
        설명 없이 키워드만 출력해.

        사용자 질문: {prompt}
        """
    )

    keyword = keyword_response.output_text.strip()

    camps = search_camping(keyword)

    if not camps:
        return "검색 결과가 없습니다. 다른 키워드로 검색해 주세요."

    camp_text = ""

    for i, camp in enumerate(camps, start=1):
        camp_text += f"""
        캠핑장 {i}
        이름: {camp.get('facltNm', '정보 없음')}
        주소: {camp.get('addr1', '정보 없음')}
        환경: {camp.get('lctCl', '정보 없음')}
        소개: {camp.get('intro', '정보 없음')}
        시설: {camp.get('sbrsCl', '정보 없음')}
        홈페이지: {camp.get('homepage', '정보 없음')}
        """

    answer = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        너는 캠핑장 추천 전문가야.

        사용자의 질문:
        {prompt}

        다음은 고캠핑 API에서 가져온 실제 검색 결과야.
        이 데이터에 없는 캠핑장이나 시설 정보를 지어내지 마.

        {camp_text}

        조건에 맞는 캠핑장을 추천하고,
        추천 이유와 주소를 한국어로 설명해 줘.
        """
    )

    return answer.output_text


prompt = input("어떤 캠핑장을 찾으시나요? ")

try:
    print(recommend_camping(prompt))
except Exception as e:
    print("오류가 발생했습니다:", e)


계곡이 있는 캠핑장을 원하신다면 다음 캠핑장을 추천합니다:

1. **증산수도계곡캠핑장**
   - 추천 이유: 푸른 산과 맑은 계곡이 어우러진 청정 자연에서 편안한 캠핑이 가능하며, 주변에 인근 문화 관광지와 먹거리 체험 공간도 많아 다양한 즐길 거리가 있습니다.
   - 주소: 경상북도 김천시 증산면 수도길 25

2. **포천계곡캠핑장**
   - 추천 이유: 약사계곡의 맑고 깨끗한 모래자갈 계곡이 있어 물놀이하기 좋고, 넓은 사이트와 프라이버시 보장, 모기가 적은 환경이 특징입니다. 특히 오후 그늘이 잘 져 시원하게 캠핑할 수 있습니다.
   - 주소: 경기도 포천시 이동면 금강로 6304

3. **각흘계곡캠핑장**
   - 추천 이유: 1km가 넘는 긴 계곡과 1급수 청정 계곡물이 매력적이며, 다양한 사이트 종류와 풍부한 그늘 덕분에 쾌적한 휴식이 가능합니다. 온수, 장작판매 등 기본 시설도 잘 갖추어져 있습니다.
   - 주소: 경기도 포천시 이동면 금강로 6439

4. **산청지막계곡캠핑장**
   - 추천 이유: 산으로 둘러싸여 공기가 좋고 깨끗한 편의시설과 넓은 자연석 강자갈 사이트를 갖추었으며, 맑고 시원한 계곡물이 아이들과 가족이 즐기기에 적합합니다.
   - 주소: 경남 산청군 금서면 왕등재로 307

위 네 곳 모두 깨끗한 계곡과 자연 환경이 뛰어나며, 시설과 주변 환경도 잘 갖추어져 있어 즐거운 계곡 캠핑을 하실 수 있습니다. 필요하시면 각각의 홈페이지를 통해 상세한 정보와 예약도 확인해 보시기 바랍니다.


In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override = True)

client = OpenAI()

# 고캠핑 API 주소
API_URL = "https://apis.data.go.kr/B551011/GoCamping/searchList"

SERVICE_KEY = os.getenv("GOCAMPING_SERVICE_KEY")


def search_camping(keyword):
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 10,
        "pageNo": 1,
        "MobileOS": "ETC",
        "MobileApp": "AppTest",
        "keyword": keyword,
        "_type": "json"
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=15
    )
    response.raise_for_status()

    data = response.json()

    items = (
        data.get("response", {})
            .get("body", {})
            .get("items", {})
            .get("item", [])
    )

    # 검색 결과가 한 건이면 리스트로 변환
    if isinstance(items, dict):
        items = [items]

    return items


def recommend_camping(prompt):
    # 질문에 맞는 검색 키워드 결정
    keyword_response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        캠핑장 검색에 사용할 핵심 키워드 하나만 출력해.
        설명 없이 키워드만 출력해.

        사용자 질문: {prompt}
        """
    )

    keyword = keyword_response.output_text.strip()

    # 고캠핑 API 검색
    camps = search_camping(keyword)

    if not camps:
        return "검색 결과가 없습니다. 다른 키워드로 검색해 주세요."

    
    camp_text = ""

    for i, camp in enumerate(camps, start=1):
        camp_text += f"""
        캠핑장 {i}
        이름: {camp.get('facltNm', '정보 없음')}
        주소: {camp.get('addr1', '정보 없음')}
        환경: {camp.get('lctCl', '정보 없음')}
        소개: {camp.get('intro', '정보 없음')}
        시설: {camp.get('sbrsCl', '정보 없음')}
        홈페이지: {camp.get('homepage', '정보 없음')}
        """

    answer = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        너는 캠핑장 추천 전문가야.

        사용자의 질문:
        {prompt}

        다음은 고캠핑 API에서 가져온 실제 검색 결과야.
        이 데이터에 없는 캠핑장이나 시설 정보를 지어내지 마.

        {camp_text}

        조건에 맞는 캠핑장을 추천하고,
        추천 이유와 주소를 한국어로 설명해 줘.
        """
    )

    return answer.output_text


prompt = input("어떤 캠핑장을 찾으시나요? ")

try:
    print(recommend_camping(prompt))
except Exception as e:
    print("오류가 발생했습니다:", e)


주말에 가족과 함께 편안하고 즐거운 시간을 보내기에 적합한 캠핑장으로 다음 세 곳을 추천드립니다.

1. **천왕산 가족캠핑장**  
   - **주소**: 서울특별시 구로구 연동로12길 149 (항동)  
   - **추천 이유**: 도심 속 자연을 만끽할 수 있고, 가족 친화적인 환경과 다양한 체험시설(북카페, 목공 체험소, 스마트 팜 농촌 체험 등)이 있어 아이들과 함께 즐겁고 유익한 시간을 보낼 수 있습니다. 또한 개별 샤워실과 온수를 제공해 쾌적한 이용이 가능하며, 산책로와 물놀이장 등 다채로운 시설이 있어 가족단위 캠핑에 매우 좋습니다.

2. **구리 토평 가족캠핑장**  
   - **주소**: 경기도 구리시 왕숙천로 11-140  
   - **추천 이유**: 도심에 위치해 접근성이 뛰어나고, 대형 연못과 연꽃이 어우러진 자연경관이 아름다워 힐링하기 좋습니다. 전기, 온수, 놀이터 등 가족이 편리하게 사용할 수 있는 시설이 잘 갖춰져 있으며, 청결하고 넓은 캠핑사이트가 마련되어 있어 쾌적한 가족 캠핑이 가능합니다.

3. **예담가족캠핑장**  
   - **주소**: 경기도 포천시 관인면 뗏마루길 251-9  
   - **추천 이유**: 자연 속에서 체험형 캠핑 문화를 즐길 수 있는 캠핑장으로, 다양한 프로그램과 이벤트가 상시 진행되어 가족 모두가 함께 즐길 거리가 많습니다. 글램핑 시설과 넓은 사이트, 편리한 마트 시설이 있어 초보자도 편안히 이용할 수 있습니다. 산, 숲, 계곡, 강 환경이 어우러져 아름다운 자연을 만끽할 수 있는 곳입니다.

이 세 곳 모두 가족이 함께 머물기에 안성맞춤인 편의시설과 다양한 즐길 거리를 갖추고 있으므로 주말 가족 나들이로 추천드립니다.


In [ ]:
keyword = '계곡'

In [37]:
from openai import OpenAI

question = '주말 가족과 놀러가려고 해 캠핑장 추천해줄래'

client = OpenAI()
response = client.responses.create(
    model = "gpt-6-luna",
    instructions = "다음 질문에서 가장 중요한 키워드 하나만 뽑아줘"
    input = question,
    max_output_tokens = 100
)
print(response.output_text)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2433967059.py, line 8)

In [40]:
url=f"http://apis.data.go.kr/B551011/GoCamping/searchList?serviceKey={gocamping_api_key}&MobileOS=ETC&MobileApp=AppTest&keyword={keyword}&_type=json"

NameError: name 'gocamping_api_key' is not defined

In [ ]:
import requests
import json
response = requests.get(url)

print(response)

NameError: name 'url' is not defined

In [39]:
data = response.json()
# print(json.dumps(data, indent=2))
sites = data['response']['body']['items']['item']
content = ""
for site in sites:
    content += f"{site['facltNm']}:{site['induty']}:{site['intro']}:{site['addr1']}"
    
print(content)

NameError: name 'response' is not defined

In [41]:
clinet = OpenAI()
response = client.responses.create(
    model = "gpt-6-luna",
    instructions = "주어진 문단을 보고 다음 질문에 대답해 줘 : {question}",
    input = content
)
print(response.output_text)

NameError: name 'content' is not defined

# LangGraph


## TypeHint

In [ ]:
@tool
def f_a(arg : int):
    return arg + 10

print(f_a(5))


15


## Pydantic

In [44]:
from pydantic import BaseModel

class MyModel(BaseModel):
    name : str
    age : int

user = MyModel(name = '홍길동', age = 30)
print(user)

name='홍길동' age=30


## LangGraph
- StateGraph

In [54]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. state 정의 처음부터 끝까지 변하는 값을 저장하고 있다.
class AgentState(TypedDict):
    messages: str

# 2. Node 정의
def first_node(state:AgentState) -> dict:
    """
    처음 노드입니다   
    """
    # state['messages'] = '안녕하세요' + state['messages']
    # return {}
    return {'messages': f"안녕하세요 + {state['messages']}"} 

def second_node(state:AgentState) -> dict:
    """
    두번째 노드입니다 
    """
    return {'messages': f"좋은 아침입니다 + {state['messages']}"}

# 3. 그래프 정의
build = StateGraph(AgentState)
build.add_node("first_node", first_node)
build.add_node("second_node", second_node)
build.add_edge(START, "first_node")
build.add_edge("first_node", "second_node")
build.add_edge("second_node", END)

# 4. 그래프 컴파일 & 실행
# build.compile()
graph = build.compile()
graph.invoke({'messages' : '홍길동'})

{'messages': '좋은 아침입니다 + 안녕하세요 + 홍길동'}

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    value: list[int]
    name: str
    result: int

def first_node(state: AgentState) -> dict:
    total = sum(state['value'])
    return {'result': total}

build = StateGraph(AgentState)
build.add_node("first_node", first_node)

build.add_edge(START, "first_node")
build.add_edge("first_node", END)

graph = build.compile()

result = graph.invoke({
    'value': [1, 2, 3],
    'name': '홍길동',
    'result': 0
})

print(result)



{'value': [1, 2, 3], 'name': '홍길동', 'result': 6}


KeyError: 'value'

In [ ]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. State 정의
class AgentState(TypedDict):
    name: str
    age: int
    msg: str

# 2. Node 정의
def first_node(state: AgentState) -> dict:
    msg = '안녕하세요 ' + state['name']
    return {'msg': msg}

def second_node(state: AgentState) -> dict:
    new_age = state['age'] + 1
    birthday_msg = f'{new_age}살 생일 축하해요'
    return {'msg': birthday_msg, 'age': new_age}

# 3. 그래프 정의
build = StateGraph(AgentState)
build.add_node("first_node", first_node)
build.add_node("second_node", second_node)

build.add_edge(START, "first_node")
build.add_edge("first_node", "second_node")
build.add_edge("second_node", END)

# 4. 그래프 컴파일 및 실행
graph = build.compile()

result = graph.invoke({
    'name': '홍길동',
    'age': 20,
    'msg': ''
})

print(result)


{'name': '홍길동', 'age': 21, 'msg': '21살 생일 축하해요'}


In [72]:
for step in graph.invoke({'name' : '홍길동', 'age': 20, 'msg': ''}):
    print(step)


name
age
msg


## conditional edges

In [ ]:
add_conditional_edges(시작, 조건router함수, 분기될 노드의 이름)

mood = 'happy' : happy_node
mood = not 'happy' : sad_node

In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. State 정의
class AgentState(TypedDict):
    mood: str
    msg: str

# 2. Node 정의
def happy_node(state: AgentState) -> dict:
    print('happy_node')
    greeting = 'i am happy'
    return {'msg': greeting}

def sad_node(state: AgentState) -> dict:
    print('sad_node')
    greeting = 'i amd sad'
    return {'msg': greeting}

def decide_mood(state:AgentState) -> Literal['happy_node', 'sad_node']:
    return 'happy_node' if state ['mood'] == 'happy' else 'sad_node'

# 3. 그래프 정의
build = StateGraph(AgentState)
build.add_node("happy_node", happy_node)
build.add_node("sad_node", sad_node)

build.add_conditional_edges(START, decide_mood)
build.add_edge("happy_node", END)
build.add_edge("sad_node", END)

# 4. 그래프 컴파일 및 실행
graph = build.compile()
result = graph.invoke({'mood': 'happy', 'msg': ''})


happy_node


In [ ]:

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. State 정의
class AgentState(TypedDict):
    mood: str
    msg: str

# 2. Node 정의
def happy_node(state: AgentState) -> dict:
    print('happy_node')
    greeting = 'i am happy'
    return {'msg': greeting}

def sad_node(state: AgentState) -> dict:
    print('sad_node')
    greeting = 'i am sad'
    return {'msg': greeting}

# 3. 기분에 따라 노드 선택
def decide_mood(state: AgentState) -> Literal['happy_node', 'sad_node']:
    return 'happy_node' if state['mood'] == 'happy' else 'sad_node'

# 4. 그래프 정의
build = StateGraph(AgentState)
build.add_node("happy_node", happy_node)
build.add_node("sad_node", sad_node)

build.add_conditional_edges(
    START,
    decide_mood,
    {
        'happy_node': 'happy_node',
        'sad_node': 'sad_node'
    }
)

build.add_edge("happy_node", END)
build.add_edge("sad_node", END)

# 5. 그래프 컴파일 및 실행
graph = build.compile()

result = graph.invoke({'mood': 'happy', 'msg': ''})
print(result)


In [ ]:

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. State 정의
class AgentState(TypedDict):
    age: int
    msg: str

# 2. Node 정의
def age_accepted(state: AgentState) -> dict:
    print('입장하세요!')
    return {'msg': '놀이기구 탑승 가능'}

def age_declined(state: AgentState) -> dict:
    print('나이가 부족하여 입장할 수 없습니다.')
    return {'msg': '놀이기구 탑승 불가'}

# 3. 키에 따라 이동할 노드 결정
def decide_age(state: AgentState) -> Literal['age_accepted', 'age_declined']:
    return 'age_accepted' if state['age'] >= 10 else 'age_declined'

# 4. 그래프 정의
build = StateGraph(AgentState)

build.add_node("age_accepted", age_accepted)
build.add_node("age_declined", age_declined)

build.add_conditional_edges(START, decide_age)
build.add_edge("age_accepted", END)
build.add_edge("age_declined", END)

# 5. 그래프 컴파일 및 실행
graph = build.compile()

result = graph.invoke({'age': 12, 'msg': ''})
print(result)


## reducer

- str : 덮어쓰기
- Annotated - 더하기

In [5]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
import operator

In [8]:
#1. state 정의 처음부터 끝까지 변하는 값을 저장하고 있다.
class AgentState(TypedDict):
    # msg: str # 덮어쓰기
    msg : Annotated[str, operator.add]

# 2. Node 정의
def happy_node(state: AgentState) -> dict:
    print('happy_node')
    greeting = 'i am happy'
    return {'msg': greeting}

def sad_node(state: AgentState) -> dict:
    print('sad_node')
    greeting = 'i am sad'
    return {'msg': greeting}

# 3. 그래프 정의
build = StateGraph(AgentState)
build.add_node("happy_node", happy_node)
build.add_node("sad_node", sad_node)
build.add_edge(START, "happy_node")
build.add_edge("happy_node", "sad_node")
build.add_edge("sad_node", END)
